In [1]:
import io
import zipfile
import xml.etree.ElementTree as ET

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
URL_FLUX = "https://donnees.roulez-eco.fr/opendata/instantane"

reponse = requests.get(URL_FLUX, timeout=120)
reponse.raise_for_status()

archive = zipfile.ZipFile(io.BytesIO(reponse.content))
print(archive.namelist())
contenu_xml = archive.read(archive.namelist()[0])
racine = ET.fromstring(contenu_xml)
print(len(racine), "stations dans le fichier")

['PrixCarburants_instantane.xml']
9827 stations dans le fichier


In [3]:
stations = []
prix = []

for pdv in racine.findall("pdv"):
    id_station = pdv.get("id")
    stations.append({
        "id": id_station,
        "latitude": pdv.get("latitude"),
        "longitude": pdv.get("longitude"),
        "cp": pdv.get("cp"),
        "type_route": pdv.get("pop"),
        "ville": pdv.findtext("ville"),
    })
    for p in pdv.findall("prix"):
        prix.append({
            "id": id_station,
            "carburant": p.get("nom"),
            "valeur": p.get("valeur"),
            "maj": p.get("maj"),
        })

df_stations = pd.DataFrame(stations)
df_prix = pd.DataFrame(prix)
print("Stations :", df_stations.shape)
print("Prix :", df_prix.shape)
df_stations.head()

Stations : (9827, 6)
Prix : (33158, 4)


,id,latitude,longitude,cp,type_route,ville
0,56580001,4806100,-269000,56580,R,Bréhan
1,89100001,4818763,330400,89100,R,SENS
2,5100009,4493400,673300,05100,R,Montgenèvre
3,6110002,4357200,701600,06110,R,Le Cannet
4,33114001,4462300,-78700,33114,R,LE BARP


In [4]:
df_prix.head()

,id,carburant,valeur,maj
0,56580001,Gazole,2.109,2026-11-12 06:55:20
1,56580001,SP95,2.009,2026-11-12 06:55:20
2,56580001,SP98,2.049,2026-11-12 06:55:21
3,89100001,Gazole,2.099,2026-11-23 14:35:41
4,89100001,E85,0.869,2026-11-10 09:49:38
